# Python Mastery — Module 3
## SQL for Data Scientists: From Basics to Analytical Queries

**Goal:** Learn to retrieve, combine, summarize, and analyze data with SQL, using a small healthcare dataset that runs locally in SQLite.

### How to use this notebook
1. Read each explanation in plain language.
2. Run code cells from top to bottom.
3. Before running a practice cell, try writing the query yourself.
4. Compare your result with the solution and explain *why* it works.

**You will learn:** `SELECT`, filtering, sorting, `NULL`, `CASE`, aggregation, `GROUP BY`, `HAVING`, joins, subqueries, CTEs, window functions, dates, data grain, and query-performance basics.

> SQLite is included with Python, so this notebook does not require a database server or extra packages.

## 1. What is SQL?

SQL (Structured Query Language) is used to ask questions of data stored in relational tables.

A table has:
- **Rows:** individual records.
- **Columns:** attributes describing each record.
- **Primary key:** a column (or set of columns) that uniquely identifies a row.
- **Foreign key:** a value that links a row to a row in another table.

A key analytics habit is to define the **grain** before querying: what does one row represent? In this notebook, one row in `claims` represents one claim line.

## 2. Create a small practice database
We'll use synthetic data only. The examples are inspired by healthcare analytics, but contain no real member information.

In [ ]:
import sqlite3
import pandas as pd

# An in-memory database disappears when the Python kernel is restarted.
conn = sqlite3.connect(":memory:")
conn.row_factory = sqlite3.Row

conn.executescript(
    """
    CREATE TABLE members (
        member_id INTEGER PRIMARY KEY,
        age INTEGER,
        gender TEXT,
        county TEXT,
        product TEXT,
        enrollment_date TEXT
    );

    CREATE TABLE claims (
        claim_id INTEGER PRIMARY KEY,
        member_id INTEGER NOT NULL,
        service_date TEXT NOT NULL,
        claim_type TEXT NOT NULL,
        place_of_service TEXT,
        medical_paid REAL DEFAULT 0,
        pharmacy_paid REAL DEFAULT 0,
        FOREIGN KEY (member_id) REFERENCES members(member_id)
    );

    CREATE TABLE providers (
        provider_id INTEGER PRIMARY KEY,
        provider_name TEXT NOT NULL,
        county TEXT
    );

    CREATE TABLE claim_provider (
        claim_id INTEGER NOT NULL,
        provider_id INTEGER NOT NULL,
        FOREIGN KEY (claim_id) REFERENCES claims(claim_id),
        FOREIGN KEY (provider_id) REFERENCES providers(provider_id)
    );
    """
)

members = [
    (101, 34, "F", "North", "Commercial", "2024-01-01"),
    (102, 67, "M", "South", "Medicare", "2023-07-01"),
    (103, 52, "F", "North", "Medicare", "2024-03-01"),
    (104, 23, "M", "West", "Commercial", "2025-01-01"),
    (105, 71, "F", "East", "Medicare", "2022-11-01"),
    (106, 45, "M", "South", "Commercial", "2024-06-01"),
]
claims = [
    (1001, 101, "2025-01-05", "Outpatient", "Clinic", 180.0, 20.0),
    (1002, 101, "2025-01-19", "Pharmacy", "Retail", 0.0, 75.0),
    (1003, 102, "2025-01-22", "Inpatient", "Hospital", 3200.0, 40.0),
    (1004, 103, "2025-02-02", "Outpatient", "Clinic", 240.0, 0.0),
    (1005, 103, "2025-02-18", "Emergency", "ED", 950.0, 10.0),
    (1006, 104, "2025-02-20", "Outpatient", "Clinic", 110.0, 0.0),
    (1007, 105, "2025-03-03", "Inpatient", "Hospital", 4100.0, 90.0),
    (1008, 106, "2025-03-11", "Pharmacy", "Retail", 0.0, 125.0),
    (1009, 102, "2025-03-19", "Emergency", "ED", 720.0, 15.0),
    (1010, 105, "2025-03-28", "Outpatient", "Clinic", 350.0, 5.0),
]
providers = [
    (1, "North Clinic", "North"),
    (2, "City Hospital", "South"),
    (3, "West Family Care", "West"),
    (4, "East Medical Center", "East"),
]
claim_provider = [
    (1001, 1), (1002, 1), (1003, 2), (1004, 1), (1005, 1),
    (1006, 3), (1007, 4), (1008, 2), (1009, 2), (1010, 4),
]

conn.executemany("INSERT INTO members VALUES (?, ?, ?, ?, ?, ?)", members)
conn.executemany("INSERT INTO claims VALUES (?, ?, ?, ?, ?, ?, ?)", claims)
conn.executemany("INSERT INTO providers VALUES (?, ?, ?)", providers)
conn.executemany("INSERT INTO claim_provider VALUES (?, ?)", claim_provider)
conn.commit()

print("Members:", pd.read_sql_query("SELECT * FROM members", conn).shape)
print("Claims:", pd.read_sql_query("SELECT * FROM claims", conn).shape)

### Helper function for displaying query results

`pd.read_sql_query()` executes a SQL query and returns the result as a DataFrame. We will use it to inspect results.

In [ ]:
def run_sql(query, params=None):
    """Run a SQL query and return its results as a pandas DataFrame."""
    return pd.read_sql_query(query, conn, params=params)

run_sql("SELECT * FROM claims LIMIT 5")

## 3. SELECT, FROM, and LIMIT

- `SELECT` chooses which columns to return.
- `FROM` tells SQL which table to read.
- `LIMIT` restricts the number of rows shown.
- An alias created with `AS` gives a column a more readable output name.

Start by selecting only the columns you need. This makes queries easier to read and can reduce unnecessary data transfer.

In [ ]:
run_sql(
    """
    SELECT claim_id, member_id, service_date, claim_type
    FROM claims
    LIMIT 5;
    """
)

## 4. Filter and sort: WHERE, ORDER BY

Common comparison operators: `=`, `!=` (or `<>`), `>`, `>=`, `<`, `<=`.

Use:
- `AND` when all conditions must be true.
- `OR` when at least one condition must be true.
- `IN (...)` for a list of values.
- `BETWEEN low AND high` for an inclusive range.
- `LIKE` for a simple text pattern.
- `ORDER BY ... DESC` for highest-to-lowest / newest-to-oldest ordering.

In [ ]:
run_sql(
    """
    SELECT claim_id, service_date, claim_type, medical_paid
    FROM claims
    WHERE medical_paid >= 500
      AND claim_type IN ('Inpatient', 'Emergency')
    ORDER BY medical_paid DESC;
    """
)

## 5. NULL values and CASE expressions

`NULL` means a value is missing or unknown. It is not the same as zero or an empty string.

- Use `IS NULL` / `IS NOT NULL`; `= NULL` does not work as expected.
- `COALESCE(value, fallback)` returns the first non-NULL value.
- `CASE WHEN ... THEN ... ELSE ... END` adds conditional logic inside a query.

In [ ]:
# Demonstrate NULL handling and conditional categories.
run_sql(
    """
    SELECT
        member_id,
        age,
        CASE
            WHEN age < 35 THEN 'Under 35'
            WHEN age < 60 THEN '35 to 59'
            ELSE '60+'
        END AS age_group,
        COALESCE(county, 'Unknown') AS county_label
    FROM members
    ORDER BY member_id;
    """
)

## 6. Aggregate functions: COUNT, SUM, AVG, MIN, MAX

Aggregates summarize multiple rows into one result:
- `COUNT(*)`: number of rows.
- `COUNT(column)`: number of non-NULL values in that column.
- `SUM(column)`: total.
- `AVG(column)`: mean.
- `MIN` / `MAX`: smallest / largest value.

A useful metric is often defined as a formula, not just a column. Here, **total cost = medical paid + pharmacy paid**.

In [ ]:
run_sql(
    """
    SELECT
        COUNT(*) AS claim_lines,
        COUNT(DISTINCT member_id) AS unique_members,
        SUM(medical_paid + pharmacy_paid) AS total_cost,
        AVG(medical_paid + pharmacy_paid) AS average_claim_cost,
        MAX(medical_paid + pharmacy_paid) AS highest_claim_cost
    FROM claims;
    """
)

## 7. GROUP BY and HAVING

`GROUP BY` creates one result row per group, such as per month or claim type.

`WHERE` filters individual rows **before** aggregation. `HAVING` filters groups **after** aggregation.

Example question: Which claim types have at least two claim lines?

In [ ]:
run_sql(
    """
    SELECT
        claim_type,
        COUNT(*) AS claim_lines,
        SUM(medical_paid + pharmacy_paid) AS total_cost
    FROM claims
    GROUP BY claim_type
    HAVING COUNT(*) >= 2
    ORDER BY total_cost DESC;
    """
)

## 8. Joins: combine related tables

- `INNER JOIN`: keep only rows with a match on both sides.
- `LEFT JOIN`: keep every row from the left table, adding matching right-side values where available.
- A join condition usually links a foreign key to a primary key.

**Critical analytics risk:** joins can multiply rows. Before joining, know the grain and relationship:
- one-to-one
- one-to-many
- many-to-many

If a member has five claims, joining members to claims should produce five rows for that member. That is expected. But summing member-level values after this join can count them five times.

In [ ]:
run_sql(
    """
    SELECT
        c.claim_id,
        c.service_date,
        m.member_id,
        m.product,
        m.county,
        c.claim_type,
        c.medical_paid + c.pharmacy_paid AS total_cost
    FROM claims AS c
    INNER JOIN members AS m
        ON c.member_id = m.member_id
    ORDER BY c.claim_id
    LIMIT 8;
    """
)

### LEFT JOIN example: retain members who have no claims

Our sample happens to have claims for every member. This query demonstrates the correct pattern and will still work if a member has no matching claim.

In [ ]:
run_sql(
    """
    SELECT
        m.member_id,
        m.product,
        COUNT(c.claim_id) AS claim_count
    FROM members AS m
    LEFT JOIN claims AS c
        ON m.member_id = c.member_id
    GROUP BY m.member_id, m.product
    ORDER BY m.member_id;
    """
)

### Join fan-out exercise

The `claim_provider` bridge table could contain multiple providers for one claim. If that happens, a claim's cost repeats on each matching row. For additive financial metrics, aggregate at the correct grain before joining or use a justified allocation rule.

In [ ]:
# Compare row counts before and after a join.
print("Claim rows:", run_sql("SELECT COUNT(*) AS n FROM claims").iloc[0, 0])
print(
    "Rows after provider join:",
    run_sql(
        """
        SELECT COUNT(*) AS n
        FROM claims c
        JOIN claim_provider cp ON c.claim_id = cp.claim_id
        JOIN providers p ON cp.provider_id = p.provider_id
        """
    ).iloc[0, 0],
)

## 9. Subqueries

A subquery is a query inside another query. It can be used in `WHERE`, `FROM`, or `SELECT`.

Use it when a question naturally has two steps—for example, first calculate average claim cost, then find claims above that average.

In [ ]:
run_sql(
    """
    SELECT
        claim_id,
        member_id,
        medical_paid + pharmacy_paid AS total_cost
    FROM claims
    WHERE medical_paid + pharmacy_paid > (
        SELECT AVG(medical_paid + pharmacy_paid)
        FROM claims
    )
    ORDER BY total_cost DESC;
    """
)

## 10. Common Table Expressions (CTEs)

A CTE is a named temporary result set defined with `WITH`. It makes a multi-step query easier to read, test, and maintain.

Think of a CTE as a named step in your analysis. It exists only for the statement that follows it.

In [ ]:
run_sql(
    """
    WITH claim_costs AS (
        SELECT
            claim_id,
            member_id,
            service_date,
            claim_type,
            medical_paid + pharmacy_paid AS total_cost
        FROM claims
    ),
    member_totals AS (
        SELECT
            member_id,
            COUNT(*) AS claim_count,
            SUM(total_cost) AS member_total_cost
        FROM claim_costs
        GROUP BY member_id
    )
    SELECT *
    FROM member_totals
    WHERE member_total_cost >= 1000
    ORDER BY member_total_cost DESC;
    """
)

## 11. Window functions

Window functions calculate values across related rows **without collapsing the rows into one row per group**.

Key examples:
- `ROW_NUMBER()`: sequential number within a group.
- `RANK()` / `DENSE_RANK()`: ranking with different tie behavior.
- `LAG()` / `LEAD()`: previous / next row's value.
- `SUM(...) OVER (...)`: running or partition-level total.

`PARTITION BY` defines groups. `ORDER BY` inside `OVER(...)` defines the order within each group.

In [ ]:
run_sql(
    """
    SELECT
        claim_id,
        member_id,
        service_date,
        medical_paid + pharmacy_paid AS total_cost,
        ROW_NUMBER() OVER (
            PARTITION BY member_id
            ORDER BY service_date
        ) AS claim_number_for_member,
        LAG(medical_paid + pharmacy_paid) OVER (
            PARTITION BY member_id
            ORDER BY service_date
        ) AS previous_claim_cost
    FROM claims
    ORDER BY member_id, service_date;
    """
)

### Rank high-cost claims

Ties matter: `RANK()` leaves gaps after ties (1, 1, 3); `DENSE_RANK()` does not (1, 1, 2). Use a deterministic tie-breaker when you need one unique row order.

In [ ]:
run_sql(
    """
    SELECT
        claim_id,
        member_id,
        medical_paid + pharmacy_paid AS total_cost,
        RANK() OVER (
            ORDER BY medical_paid + pharmacy_paid DESC
        ) AS cost_rank
    FROM claims
    ORDER BY cost_rank, claim_id;
    """
)

## 12. Date-based analytics

SQLite stores our dates as ISO-formatted text (`YYYY-MM-DD`). That format sorts chronologically as text, but use date functions for date transformations.

- `strftime('%Y-%m', service_date)` extracts a year-month label.
- Filter dates with a clear inclusive/exclusive convention. A common robust pattern is `date >= start AND date < next_period_start`.
- Date functions vary across database systems; SQL Server, PostgreSQL, BigQuery, and Snowflake have different syntax.

In [ ]:
run_sql(
    """
    SELECT
        strftime('%Y-%m', service_date) AS service_month,
        COUNT(*) AS claim_lines,
        COUNT(DISTINCT member_id) AS unique_members,
        SUM(medical_paid) AS medical_cost,
        SUM(pharmacy_paid) AS pharmacy_cost,
        SUM(medical_paid + pharmacy_paid) AS total_cost
    FROM claims
    GROUP BY strftime('%Y-%m', service_date)
    ORDER BY service_month;
    """
)

## 13. Analytical pattern: top members by total cost

This query first summarizes claims to one row per member, then joins the member dimension to add descriptive attributes. That is safer than mixing raw claim rows with member-level metrics.

In [ ]:
run_sql(
    """
    WITH member_costs AS (
        SELECT
            member_id,
            COUNT(*) AS claim_count,
            SUM(medical_paid + pharmacy_paid) AS total_cost
        FROM claims
        GROUP BY member_id
    )
    SELECT
        mc.member_id,
        m.product,
        m.county,
        mc.claim_count,
        mc.total_cost
    FROM member_costs AS mc
    JOIN members AS m
        ON mc.member_id = m.member_id
    ORDER BY mc.total_cost DESC
    LIMIT 5;
    """
)

## 14. Query performance basics

A practical checklist:
1. Select only the columns needed; avoid `SELECT *` in production queries.
2. Filter early when it reduces the data being processed.
3. Join on appropriate keys and confirm expected row counts.
4. Avoid wrapping indexed filter columns in functions when it prevents index use; use a date range where possible.
5. Consider indexes on frequently filtered or joined columns, but remember indexes consume storage and add write overhead.
6. Use `EXPLAIN QUERY PLAN` in SQLite to inspect the query strategy.
7. Benchmark on representative data; tiny examples do not predict production performance.

In [ ]:
# An index can help repeated lookups by member_id.
conn.execute("CREATE INDEX idx_claims_member_id ON claims(member_id)")
conn.commit()

plan = run_sql(
    """
    EXPLAIN QUERY PLAN
    SELECT claim_id, service_date
    FROM claims
    WHERE member_id = 102;
    """
)
plan

## 15. Practice set — try before opening the solutions

Write a SQL query for each task.

1. Return all claims from February 2025, oldest first.
2. Calculate total medical, pharmacy, and combined cost by `claim_type`.
3. Find members whose total claim cost exceeds 1,000. Show each member once.
4. Show monthly total cost by product (Commercial vs Medicare).
5. For each member, number their claims from newest to oldest using `ROW_NUMBER()`.
6. Find the highest-cost claim for each member.
7. Keep every member in the output, including anyone with zero claims.
8. Explain why joining a claim to multiple provider bridge rows can inflate summed claim cost.

### Practice 1 — February claims

In [ ]:
# TODO: Write your query in the triple-quoted string.
practice_1 = """
SELECT claim_id, member_id, service_date, claim_type
FROM claims
WHERE service_date >= '2025-02-01'
  AND service_date < '2025-03-01'
ORDER BY service_date;
"""
run_sql(practice_1)

### Practice 2 — Cost by claim type

In [ ]:
practice_2 = """
SELECT
    claim_type,
    SUM(medical_paid) AS medical_cost,
    SUM(pharmacy_paid) AS pharmacy_cost,
    SUM(medical_paid + pharmacy_paid) AS total_cost
FROM claims
GROUP BY claim_type
ORDER BY total_cost DESC;
"""
run_sql(practice_2)

### Practice 3 — Members above $1,000 total cost

In [ ]:
practice_3 = """
SELECT member_id, SUM(medical_paid + pharmacy_paid) AS total_cost
FROM claims
GROUP BY member_id
HAVING SUM(medical_paid + pharmacy_paid) > 1000
ORDER BY total_cost DESC;
"""
run_sql(practice_3)

### Practice 4 — Monthly cost by product

In [ ]:
practice_4 = """
SELECT
    strftime('%Y-%m', c.service_date) AS service_month,
    m.product,
    SUM(c.medical_paid + c.pharmacy_paid) AS total_cost
FROM claims c
JOIN members m ON c.member_id = m.member_id
GROUP BY strftime('%Y-%m', c.service_date), m.product
ORDER BY service_month, m.product;
"""
run_sql(practice_4)

### Practice 5 — Number claims newest to oldest per member

In [ ]:
practice_5 = """
SELECT
    claim_id,
    member_id,
    service_date,
    ROW_NUMBER() OVER (
        PARTITION BY member_id
        ORDER BY service_date DESC, claim_id DESC
    ) AS newest_first_number
FROM claims
ORDER BY member_id, newest_first_number;
"""
run_sql(practice_5)

### Practice 6 — Highest-cost claim per member

In [ ]:
practice_6 = """
WITH ranked_claims AS (
    SELECT
        claim_id,
        member_id,
        service_date,
        medical_paid + pharmacy_paid AS total_cost,
        ROW_NUMBER() OVER (
            PARTITION BY member_id
            ORDER BY medical_paid + pharmacy_paid DESC, claim_id
        ) AS rn
    FROM claims
)
SELECT claim_id, member_id, service_date, total_cost
FROM ranked_claims
WHERE rn = 1
ORDER BY total_cost DESC;
"""
run_sql(practice_6)

### Practice 7 — All members, including members without claims

In [ ]:
practice_7 = """
SELECT
    m.member_id,
    COUNT(c.claim_id) AS claim_count,
    COALESCE(SUM(c.medical_paid + c.pharmacy_paid), 0) AS total_cost
FROM members m
LEFT JOIN claims c ON m.member_id = c.member_id
GROUP BY m.member_id
ORDER BY m.member_id;
"""
run_sql(practice_7)

## 16. Quick knowledge check

Answer in your own words before reading the answer key:
- What is the difference between `WHERE` and `HAVING`?
- When would you use a `LEFT JOIN` rather than an `INNER JOIN`?
- Why is `COUNT(*)` different from `COUNT(column)` when NULL values exist?
- What does `PARTITION BY` do in a window function?
- Why can a join cause total cost to be overstated?

### Suggested answer key
- `WHERE` filters input rows; `HAVING` filters aggregated groups.
- `LEFT JOIN` retains every left-table row, even if no match exists on the right.
- `COUNT(*)` counts rows; `COUNT(column)` counts only non-NULL values in that column.
- `PARTITION BY` splits rows into independent groups for the window calculation without collapsing the rows.
- A one-to-many or many-to-many join can repeat the same cost across several output rows; aggregate at the correct grain or apply a justified allocation rule.

## 17. Final mini-project — build a monthly healthcare cost report

**Business question:** How are medical and pharmacy costs changing by month, product, and county?

Deliver:
1. Monthly cost summary by product and county.
2. Member-level total cost and claim count.
3. Top 5 highest-cost members.
4. A query that identifies each member's most expensive claim.
5. A short interpretation with at least two observations and one data-quality caveat.

**Suggested SQL query for deliverable 1:**

In [ ]:
monthly_report_sql = """
SELECT
    strftime('%Y-%m', c.service_date) AS service_month,
    m.product,
    m.county,
    COUNT(*) AS claim_lines,
    COUNT(DISTINCT c.member_id) AS unique_members,
    SUM(c.medical_paid) AS medical_cost,
    SUM(c.pharmacy_paid) AS pharmacy_cost,
    SUM(c.medical_paid + c.pharmacy_paid) AS total_cost
FROM claims c
JOIN members m ON c.member_id = m.member_id
GROUP BY
    strftime('%Y-%m', c.service_date),
    m.product,
    m.county
ORDER BY service_month, m.product, m.county;
"""
monthly_report = run_sql(monthly_report_sql)
monthly_report

## 18. Readiness checklist

You are ready to move on when you can do these without copying the solutions:
- [ ] Select, filter, sort, and limit rows.
- [ ] Handle `NULL` correctly and write a `CASE` expression.
- [ ] Summarize with `COUNT`, `SUM`, and `AVG`.
- [ ] Explain `WHERE` versus `HAVING`.
- [ ] Join tables and verify the expected row count.
- [ ] Use a CTE to break a complex query into steps.
- [ ] Use `ROW_NUMBER`, `RANK`, and `LAG`.
- [ ] Aggregate data by month and category.
- [ ] Explain how join fan-out can inflate a metric.
- [ ] Describe one way to inspect query performance.

### Next module
**Module 4 — Statistics and Probability for Data Science:** distributions, sampling, confidence intervals, hypothesis testing, p-values, effect sizes, correlation, regression foundations, and common interpretation mistakes.